# 🍛 CulinaryVLM — Fine-tuned Model Evaluation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shivam697/CulinaryVLM/blob/main/notebooks/evaluate_vlm.ipynb)

Verifies **[shivamminde/culinary-vlm-qlora](https://huggingface.co/shivamminde/culinary-vlm-qlora)** — QLoRA adapter on Llama-3.2-11B-Vision for Indian cooking QA.

**Before running:** `Runtime → Change runtime type → T4 GPU`

| Detail | Value |
|---|---|
| Base model | `unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit` |
| Adapter | QLoRA r=16, alpha=32 — language layers only |
| Vision layers | **Frozen** — text-in, text-out at inference |
| Input format | `Category + Context + Question → Answer` |

## Step 1 — Install Dependencies

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q huggingface_hub
print('✅ Done')

## Step 2 — Authenticate with Hugging Face
Add token as Colab secret `HF_TOKEN` (🔑 icon in left sidebar) or paste below.

In [ ]:
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
    print('✅ Logged in via Colab secret HF_TOKEN')
except Exception:
    login()
    print('✅ Logged in interactively')

## Step 3 — Load the Fine-Tuned Model

> `culinary-vlm-qlora` is a **LoRA adapter** — Unsloth loads base + adapter together.  
> We pin everything to `cuda:0` with `device_map={"":0}` to avoid device-split issues.

In [ ]:
from unsloth import FastVisionModel
import torch

ADAPTER_ID = "shivamminde/culinary-vlm-qlora"

print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Loading {ADAPTER_ID} ...")

# device_map={"":0} forces ALL layers onto cuda:0.
# Avoids 'RuntimeError: tensors on different devices' when device_map="auto"
# splits layers between CPU and GPU.
model, processor = FastVisionModel.from_pretrained(
    model_name=ADAPTER_ID,
    max_seq_length=2048,
    dtype=torch.bfloat16,
    load_in_4bit=True,
    device_map={"":0},
)
FastVisionModel.for_inference(model)

# Extract inner text tokenizer from the MllamaProcessor.
# Calling processor(string) triggers image loading → ValueError.
text_tokenizer = processor.tokenizer if hasattr(processor, 'tokenizer') else processor

# Determine actual model device for input placement
MODEL_DEVICE = next(model.parameters()).device
print(f'\n✅ Model ready on {MODEL_DEVICE}')
print(f'Processor : {type(processor).__name__}')
print(f'Tokenizer : {type(text_tokenizer).__name__}')

## Step 4 — Single Query Inference

In [ ]:
def ask_model(category: str, question: str, context: str = "") -> str:
    """Text-only QA inference matching the training input format."""
    # Build prompt — same format as finetune_qlora.py → format_qa_for_training()
    parts = [f"Category: {category}"]
    if context:
        parts.append(f"Context: {context}")
    parts.append(f"Question: {question}")
    user_text = "\n\n".join(parts)

    # Use text_tokenizer (inner tokenizer), NOT processor.
    # processor(string) → MllamaProcessor image loading → ValueError.
    messages = [{"role": "user", "content": user_text}]
    prompt = text_tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    # Move inputs to the same device as the model (MODEL_DEVICE = cuda:0)
    inputs = text_tokenizer(
        prompt, return_tensors="pt", add_special_tokens=False
    ).to(MODEL_DEVICE)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            use_cache=True,
        )

    generated_ids = output[0][inputs["input_ids"].shape[1]:]
    return text_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


# ── Example 1: Segment-grounded QA (with context) ───────────────────────────
a1 = ask_model(
    category="Hyderabadi",
    context="Action: Adding turmeric powder. Description: The person is adding turmeric powder to the marinated chicken in the bowl.",
    question="What ingredients are being used in this step?"
)
print("Q1: What ingredients are being used in this step?")
print(f"A1: {a1}\n")

# ── Example 2: General knowledge (no context) ───────────────────────────────
a2 = ask_model(
    category="Kolkata",
    question="What makes Kolkata biryani different from Hyderabadi biryani?"
)
print("Q2: What makes Kolkata biryani different from Hyderabadi biryani?")
print(f"A2: {a2}\n")

# ── Example 3: Technique question ────────────────────────────────────────────
a3 = ask_model(
    category="Lucknowi",
    context="Action: Dum cooking. Description: The pot is sealed with dough and placed over low heat.",
    question="Why is the pot sealed with dough?"
)
print("Q3: Why is the pot sealed with dough?")
print(f"A3: {a3}")

## Step 5 — Batch Evaluation on Test Set

In [ ]:
import os

if not os.path.exists("test_filtered.json"):
    print("Cloning test set from GitHub...")
    !git clone --depth 1 --filter=blob:none --sparse https://github.com/shivam697/CulinaryVLM.git _repo 2>/dev/null
    !cd _repo && git sparse-checkout set datasets/qa 2>/dev/null
    !cp _repo/datasets/qa/test_filtered.json test_filtered.json
    print('✅ test_filtered.json ready')
else:
    print('✅ Already present')

In [ ]:
import json

with open("test_filtered.json") as f:
    raw = json.load(f)

test_data = [
    qa for qa in raw.get("qa_pairs", [])
    if not qa.get("needs_generation")
    and qa.get("question", "").strip()
    and qa.get("answer", "").strip()
]
print(f"Valid test samples: {len(test_data)}")

N = 20
scores = []

for i, qa in enumerate(test_data[:N]):
    category = qa.get("category", "biryani")
    context  = qa.get("context", "")
    question = qa.get("question", "")
    expected = qa.get("answer", "")

    generated = ask_model(category=category, context=context, question=question)

    exp_words = set(expected.lower().split())
    gen_words = set(generated.lower().split())
    overlap = len(exp_words & gen_words) / max(len(exp_words), 1)
    scores.append(overlap)

    print(f"[{i+1}/{N}] {category}")
    print(f"  Q        : {question[:80]}")
    print(f"  Expected : {expected[:100]}")
    print(f"  Generated: {generated[:100]}")
    print(f"  Overlap  : {overlap:.0%}\n")

avg = sum(scores) / len(scores) if scores else 0
above_30 = sum(1 for s in scores if s >= 0.3)
print(f"{'='*60}")
print(f"Avg keyword overlap    : {avg:.0%}")
print(f"Samples with ≥30% match: {above_30}/{N} ({above_30/N:.0%})")

---
## ✅ Done!

| | Link |
|---|---|
| 🌐 Live demo | [culinary-vlm.vercel.app](https://culinary-vlm.vercel.app) |
| ⚙️ API | [culinary-vlm-api.onrender.com](https://culinary-vlm-api.onrender.com) |
| 🤗 Model | [shivamminde/culinary-vlm-qlora](https://huggingface.co/shivamminde/culinary-vlm-qlora) |
| 📦 Repo | [github.com/shivam697/CulinaryVLM](https://github.com/shivam697/CulinaryVLM) |